In [ ]:
# 读取指定目录下的文件夹dataset4geodiff\raw_geodiff_txt，每一个文件夹的名字就是apkname，然后每一个文件夹内有version_mapping.json, 我需要读取这个json的内容，第一个value是版本号1，第二个value是版本号2.
# 然后加载这份文件dataset4geodiff\androzoo_gfd_app_metadata.tsv，这个是每一个apkname在不同地区对应的版本号。

# 最终我希望构建这个apk在不同国家的

## 读取指定目录下的文件夹dataset4geodiff\out_openai_sematic_geodiff_txt，每一个文件夹的名字就是apkname，打开该文件夹。
然后加载这份文件dataset4geodiff\androzoo_gfd_app_metadata.tsv，这个是每一个apkname在不同地区对应的版本号。

最终我希望在该文件夹目录下得到该apk在不同国家的版本号，保存为一个csv就行，一个apkname文件夹对应一个输出

遍历 raw_geodiff_txt 下每个 apk 文件夹

用文件夹名作为 apkname 去 androzoo_gfd_app_metadata.tsv 匹配

在每个 apk 文件夹里各自输出一个 country_versions.csv

In [5]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA4_forth_100_batch" # AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

# {parameter}
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1
parameter = "first_6"

In [6]:
from pathlib import Path
import pandas as pd
import json

# first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1

raw_dir = Path(f"dataset4geodiff/out_openai_sematic_geodiff_txt/{parameter}")
meta_path = Path("dataset4geodiff/androzoo_gfd_app_metadata.tsv")

In [7]:
# 读取 metadata
meta_df = pd.read_csv(meta_path, sep="\t", dtype=str).fillna("")
meta_df.columns = [c.strip() for c in meta_df.columns]
meta_df["package_name"] = meta_df["package_name"].astype(str).str.strip()

In [8]:
# 只保留国家版本相关列
country_cols = [
    c for c in meta_df.columns
    # if c == "package_name" or c.endswith("_version_name") or c.endswith("_version_code")
    if c == "package_name" or c.endswith("_version_code")
]

apk_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])

matched = 0
unmatched = 0

for apk_dir in apk_dirs:
    apkname = apk_dir.name.strip()

    # skip folders that are not APK package names
    if "." not in apkname:
        print(f"[SKIP] Not an APK folder: {apkname}")
        continue

    print(f"处理 apk: {apkname}")
    # 匹配该 apk 的 metadata 行
    row = meta_df.loc[meta_df["package_name"] == apkname, country_cols].copy()

    if row.empty:
        unmatched += 1
        # 未匹配也输出一个文件，方便排查
        row = pd.DataFrame([{"package_name": apkname}])
        for c in country_cols:
            if c != "package_name":
                row[c] = ""
    else:
        matched += len(row)

    # 可选：把 version_mapping.json 的前两个 value 也写进去
    # vm_path = apk_dir / "version_mapping.json"
    # v1, v2 = "", ""
    # if vm_path.exists():
    #     try:
    #         vm_obj = json.loads(vm_path.read_text(encoding="utf-8"))
    #         vals = list(vm_obj.values())
    #         if len(vals) > 0:
    #             v1 = str(vals[0])
    #         if len(vals) > 1:
    #             v2 = str(vals[1])
    #     except Exception:
    #         pass

    # row["version_1_from_mapping"] = v1
    # row["version_2_from_mapping"] = v2

    # 每个 apk 文件夹一个输出
    out_csv = apk_dir / f"{apkname}_country_versions.csv"
    row.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"已写出: {out_csv}")

print(f"apk 文件夹总数: {len(apk_dirs)}")
print(f"匹配到 metadata 的行数: {matched}")
print(f"未匹配 apk 数: {unmatched}")

apk 文件夹总数: 0
匹配到 metadata 的行数: 0
未匹配 apk 数: 0



## 将dataset4geodiff\EPP-LLM_ours_dataset_mapping.xlsx转为clauses.json：(dont need run again)
[
    
  {"ours_id": "P1", "bucket": "P", "title": "Categories of Personal Data Collected"},

  {"ours_id": "P2", "bucket": "P", "title": "Categories of PI Shared"},

In [ ]:
# import pandas as pd
# import json
# from pathlib import Path
# import re

# clauses = [
#     {"ours_id": "P1", "bucket": "P", "title": "Categories of Personal Data Collected"},
#     {"ours_id": "P2", "bucket": "P", "title": "Categories of PI Shared"},
#     # 继续追加其它条例...
# ]

# xlsx_path = Path("dataset4geodiff/EPP-LLM_ours_dataset_mapping.xlsx")
# out_path = Path("dataset4geodiff/clauses.json")  # 需要的话改成你的目标路径

# # 读 Excel（默认第一个 sheet）
# df = pd.read_excel(xlsx_path)
# df.columns = [str(c).strip() for c in df.columns]

# # 统一列名（兼容不同命名）
# lower_to_raw = {c.lower(): c for c in df.columns}

# def pick_col(candidates):
#     for c in candidates:
#         if c.lower() in lower_to_raw:
#             return lower_to_raw[c.lower()]
#     return None

# col_id = pick_col(["ours_id"])
# # col_bucket = pick_col(["bucket", "group", "category"])
# col_title = pick_col(["clause"])

# if col_id is None or col_title is None:
#     raise ValueError(f"找不到必要列。当前列有: {df.columns.tolist()}")

# # 清洗
# df = df[[col_id, col_title]].copy()
# df[col_id] = df[col_id].astype(str).str.strip()
# df[col_title] = df[col_title].astype(str).str.strip()

# # bucket 直接由 ours_id 前缀提取
# # P1 -> P, P2 -> P, CR6 -> CR, E24 -> E
# df["bucket"] = df[col_id].str.extract(r"^([A-Za-z]+)", expand=False).fillna("").str.upper()

# # 去空、去重
# df = df[(df[col_id] != "") & (df[col_title] != "") & (df["bucket"] != "")]
# df = df.drop_duplicates(subset=[col_id], keep="first")

# clauses = [
#     {"ours_id": r[col_id], "bucket": r["bucket"], "title": r[col_title]}
#     for _, r in df.iterrows()
# ]

# out_path.parent.mkdir(parents=True, exist_ok=True)
# out_path.write_text(json.dumps(clauses, ensure_ascii=False, indent=2), encoding="utf-8")

# print("已保存:", out_path)
# print("总条数:", len(clauses))
# print("前2条示例:")
# print(json.dumps(clauses[:2], ensure_ascii=False, indent=2))

已保存: dataset4geodiff\clauses.json
总条数: 88
前2条示例:
[
  {
    "ours_id": "P1",
    "bucket": "P",
    "title": "Categories of Personal Data Collected"
  },
  {
    "ours_id": "P2",
    "bucket": "P",
    "title": "Categories of PI Shared"
  }
]


## 不用

<!-- ## 把每条 path diff 解释转成下面这种结构化记录

  {
  "app_info": {
    "app_id": {{app_id}},
    "baseline_version": "string",
    "target_version": "string",
    "diff_direction": "target_minus_baseline"
    },
   "path_changes": [
        {
        "PATH_CHANGE_ID": "...",
        "CHANGE_SUMMARY": "...",
        "SUPPORTING_EVIDENCE": [...],
        "SECURITY_IMPLICATIONS": [...],
        "OPTIONAL_APP_SUMMARY": "..."
        },
        {
        "PATH_CHANGE_ID": "...",
        "CHANGE_SUMMARY": "...",
        "SUPPORTING_EVIDENCE": [...],
        "SECURITY_IMPLICATIONS": [...],
        "OPTIONAL_APP_SUMMARY": "..."
        }
    ]
} -->


In [ ]:
# 遍历dataset4geodiff\raw_geodiff_txt目录下的每个apk目录，目录名字就是apkname，读取其中以country_versions.csv结尾文件，虽然有10个地区，但是versioncode只有2个不同的版本号，baseline_version是usa_version_code，

# 加载dataset4geodiff\apk_versions_long.csv这个文件，找到这个apkname对应的versioncode；target_version是这里面除了baseline_version以外的另一个版本号。

# 构建下面格式的json文件输出到dataset4geodiff\processed_outputs目录下，文件名为{app_id}_processed_converted.json。

#  {
#      "app_id": "{{app_id}}",
#      "app_info": {
#          "baseline_version": "string",
#          "target_version": "string",
#          "diff_direction": "target_minus_baseline"
#      },
#      "path_changes": [
#          {
#          "PATH_CHANGE_ID": "...",
#          "CHANGE_SUMMARY": "...",
#          "SUPPORTING_EVIDENCE": [...],
#          "SECURITY_IMPLICATIONS": [...],
#          "OPTIONAL_APP_SUMMARY": "..."
#          },
#          {
#          "PATH_CHANGE_ID": "...",
#          "CHANGE_SUMMARY": "...",
#          "SUPPORTING_EVIDENCE": [...],
#          "SECURITY_IMPLICATIONS": [...],
#          "OPTIONAL_APP_SUMMARY": "..."
#          }
#      ]

In [2]:
# import json
# from pathlib import Path
# import pandas as pd

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# all_apks_info = pd.read_csv("dataset4geodiff/apk_versions_long.csv", dtype=str)
# all_apks_info.head(), all_apks_info.shape

(                    apkname  version
 0        HinKhoj.Dictionary      310
 1        HinKhoj.Dictionary      320
 2  ae.brandsforless.android      412
 3  ae.brandsforless.android      413
 4       air.bg.lan.Monopoli  7000009,
 (2193, 2))

In [ ]:
# RAW_DIR = Path("dataset4geodiff/raw_geodiff_txt")
# # CHANGE_DIR = Path("dataset4geodiff/out_openai_sematic_geodiff_txt")
# OUT_DIR = Path("dataset4geodiff/out_openai_sematic_geodiff_txt")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# def pick_target_version(row: pd.Series, baseline: str) -> str:
#     # 所有 *_version_code 列里，挑一个非 usa 且与 baseline 不同的版本
#     code_cols = [c for c in row.index if c.endswith("_version_code")]
#     vals = []
#     for c in code_cols:
#         if c == "usa_version_code":
#             continue
#         v = str(row.get(c, "")).strip()
#         if v:
#             vals.append(v)

#     # 去重并排除 baseline
#     uniq = []
#     for v in vals:
#         if v not in uniq:
#             uniq.append(v)
#             uniq = [v for v in uniq if v != baseline]

#     return uniq[0] if uniq else ""

# def find_target_version(all_info_df: pd.DataFrame, app_id: str, baseline: str) -> str:
#     # 从所有 apk 版本信息里，找到一个与 baseline 不同的版本
#     apk = all_info_df[all_info_df["apkname"] == app_id]
#     if apk.empty:
#         return ""
#     vals = apk["version"].astype(str).str.strip().unique().tolist()
#     uniq = []
#     for v in vals:
#         if v not in uniq:
#             if v != baseline:
#                 uniq.append(v)
#     print(app_id,uniq)
#     return uniq[0] if uniq else ""

In [11]:
# ok_count = 0
# skip_count = 0

# for apk_dir in sorted([p for p in RAW_DIR.iterdir() if p.is_dir()]):
#     app_id = apk_dir.name.strip()

#     # 1) 读取 country_versions.csv
#     csv_files = list(apk_dir.glob("*_country_versions.csv"))
#     if not csv_files:
#         print(f"[跳过] {app_id}: 未找到 *_country_versions.csv")
#         skip_count += 1
#         continue

#     csv_path = csv_files[0]
#     try:
#         df = pd.read_csv(csv_path, dtype=str).fillna("")
#     except Exception as e:
#         print(f"[跳过] {app_id}: 读取 csv 失败 -> {e}")
#         skip_count += 1
#         continue

#     if df.empty:
#         print(f"[跳过] {app_id}: csv 为空")
#         skip_count += 1
#         continue

#     row = df.iloc[0]
#     baseline_version = str(row.get("usa_version_code", "")).strip()
#     target_version = find_target_version(all_apks_info, app_id, baseline_version)

#     # target_version = pick_target_version(row, baseline_version)

#     # 2) 读取差异语义文件
#     change_path = OUT_DIR / app_id /f"{app_id}_processed.json"
#     # print(change_path)
#     if not change_path.exists():
#         print(f"[跳过] {app_id}: 未找到差异文件 {change_path.name}")
#         skip_count += 1
#         continue

#     try:
#         src = json.loads(change_path.read_text(encoding="utf-8"))
#     except Exception as e:
#         print(f"[跳过] {app_id}: 读取差异 json 失败 -> {e}")
#         skip_count += 1
#         continue

#     changes = src.get("changes", [])
#     app_summary = src.get("changes_summarization", {}).get("summary", "")

#     # 3) 构建目标结构
#     out_obj = {
#         "app_id": app_id,
#         "app_info": {
#             "baseline_version": baseline_version,
#             "target_version": target_version,
#             "diff_direction": "target_minus_baseline"
#         },
#         "path_changes": []
#     }

#     for idx, change in enumerate(changes, start=1):
#         out_obj["path_changes"].append({
#             "PATH_CHANGE_ID": f"PC{idx:03d}",
#             "CHANGE_SUMMARY": change.get("change_summary", ""),
#             "SUPPORTING_EVIDENCE": change.get("supporting_evidence", []),
#             "SECURITY_IMPLICATIONS": change.get("security_implications", []),
#             "OPTIONAL_APP_SUMMARY": app_summary
#         })
#     print(out_obj)
#     # 4) 写出
#     out_path = OUT_DIR / app_id /f"{app_id}_processed_converted.json"
#     out_path.write_text(json.dumps(out_obj, ensure_ascii=False, indent=2), encoding="utf-8")
#     print(f"[OK] {app_id} -> {out_path}")
#     ok_count += 1
#     print(f"\n完成: 成功 {ok_count}, 跳过 {skip_count}")

com.bandagames.mpuzzle.gp ['747']
com.benoitletondor.pixelminimalwatchface ['10226']


## 将结果存为json格式

In [ ]:
# rs2 = '''
# {
# "app_id": "com.benoitletondor",
# "path_results": [
# {
# "path_change_id": "PC_001",
# "privacy_relevance": "none",
# "evidence_strength": "none",
# "event_types": [
# "NO_PRIVACY_RELEVANT_SIGNAL"
# ],
# "actions": [
# "none"
# ],
# "data_categories": [
# "none"
# ],
# "recipients": [
# "none"
# ],
# "purposes": [
# "none"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.95,
# "reasoning": "This path reflects removal of dynamic UI composition steps in MainActivity.onCreate, leaving only window access. The evidence points to startup/interface simplification rather than collection, processing, storage, sharing, consent, or rights-related behavior.",
# "uncertainties": [
# "The removed UI path could previously have supported privacy-relevant prompts or notices, but no such purpose is shown in the evidence.",
# "getWindow alone is not a meaningful privacy signal in this context."
# ]
# },
# {
# "path_change_id": "PC_002",
# "privacy_relevance": "medium",
# "evidence_strength": "strong_indirect",
# "event_types": [
# "EXTERNAL_ROUTING_SIGNAL",
# "NOTIFICATION_ACCESS_SIGNAL",
# "DEVICE_STATE_ACCESS_SIGNAL"
# ],
# "actions": [
# "route_to_external_component",
# "access_notification",
# "access_device_state"
# ],
# "data_categories": [
# "notification_content",
# "battery_status"
# ],
# "recipients": [
# "other_app",
# "operating_system",
# "unknown_external_recipient"
# ],
# "purposes": [
# "app_functionality",
# "system_integration",
# "unknown"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.7,
# "reasoning": "The old configuration/settings-driven startup flow was replaced with an asynchronous intent- and URI-based routing flow using chooser/package/class/data/extras APIs. References to notification-listener and battery-status receiver components provide indirect support that the new path may involve notification-related information and device-state integration. The evidence supports privacy-relevant routing and possible access signals, but it does not directly prove concrete personal-data collection, transfer, or disclosure content.",
# "uncertainties": [
# "Intent extras and URIs are present, but their payloads are unknown, so concrete data categories beyond indirect notification and battery/device-state signals cannot be confirmed.",
# "The notification-listener and battery-status references may be only component linkage rather than actual user-data use in this path.",
# "It is unclear whether routing stays fully internal, targets the operating system, or invokes third-party apps/services."
# ]
# },
# {
# "path_change_id": "PC_003",
# "privacy_relevance": "weak",
# "evidence_strength": "weak_indirect",
# "event_types": [
# "DATA_STORAGE_SIGNAL",
# "EXTERNAL_ROUTING_SIGNAL"
# ],
# "actions": [
# "store",
# "route_to_external_component"
# ],
# "data_categories": [
# "unknown_personal_data"
# ],
# "recipients": [
# "controller"
# ],
# "purposes": [
# "support",
# "app_functionality"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.56,
# "reasoning": "This path removes support-screen interactivity including startActivity/result passing and local storage-related APIs such as cache, no-backup files, and database helper configuration. That gives a weak indirect signal of reduced local data handling and reduced support-flow routing, but the evidence does not show what data, if any, was handled and does not directly establish privacy-specific operations.",
# "uncertainties": [
# "The storage-related APIs may have been used for non-personal UI/support artifacts rather than personal data.",
# "The removed activity/result paths may reflect generic navigation rather than privacy-relevant external routing.",
# "No direct evidence identifies any specific support request data, account data, or contact data."
# ]
# },
# {
# "path_change_id": "PC_004",
# "privacy_relevance": "none",
# "evidence_strength": "none",
# "event_types": [
# "NO_PRIVACY_RELEVANT_SIGNAL"
# ],
# "actions": [
# "none"
# ],
# "data_categories": [
# "none"
# ],
# "recipients": [
# "none"
# ],
# "purposes": [
# "none"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.92,
# "reasoning": "The evidence shows splash/relaunch lifecycle simplification, removed UI traversal, and removal of timer cancellation on stop. No direct or indirect signal of personal-data collection, processing, storage, sharing, consent handling, or user-right handling is supported by these path changes.",
# "uncertainties": [
# "Lifecycle changes could theoretically affect background behavior duration, but no privacy-relevant data operation is evidenced here.",
# "The helper-branch substitution in splash initialization is too abstract to support a privacy interpretation."
# ]
# }
# ]
# }
# '''

In [ ]:
# import json
# from pathlib import Path

# # rs 是你现在那个三引号字符串
# data = json.loads(rs2)

# input_file = Path("dataset4geodiff\out_openai_sematic_geodiff_txt\com.benoitletondor.pixelminimalwatchface\com.benoitletondor.pixelminimalwatchface_processed_converted.json")         # 改成你的原始文件路径
# # print(input_file.parent)
# app_id = input_file.stem.replace("_processed_converted", "")  # 从输入文件名

# out_path = Path(f"{input_file.parent}/{app_id}_prompt1_output.json")
# # out_path.parent.mkdir(parents=True, exist_ok=True)

# with out_path.open("w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)

# print("已保存:", out_path)

In [ ]:
# rs = '''
# {
# "app_id": "com.bandagames.mpuzzle.gp",
# "path_results": [
# {
# "path_change_id": "PC001",
# "privacy_relevance": "none",
# "evidence_strength": "none",
# "event_types": [
# "NO_PRIVACY_RELEVANT_SIGNAL"
# ],
# "actions": [
# "none"
# ],
# "data_categories": [
# "none"
# ],
# "recipients": [
# "none"
# ],
# "purposes": [
# "none"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.94,
# "reasoning": "This path shows removal of launch-time initialization/parsing logic and removal of a generic getSystemService access during MainActivity startup. The available evidence supports startup simplification only; it does not identify any concrete personal-data collection, processing, storage, sharing, consent flow, rights handling, or specific system-service use with privacy significance.",
# "uncertainties": [
# "The removed system service is not identified, so its function cannot be characterized beyond generic startup access.",
# "The removed parsing/setup flow could have affected configuration behavior, but no privacy-specific content or data handling is shown.",
# "The change occurs at app launch, but no direct notice, consent, permission, or user-data operation is evidenced."
# ]
# }
# ]
# }


# '''

In [ ]:
# import json
# from pathlib import Path

# # rs 是你现在那个三引号字符串
# data = json.loads(rs)

# input_file = Path("dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp\com.bandagames.mpuzzle.gp_processed_converted.json")         # 改成你的原始文件路径
# print(input_file.parent)
# app_id = input_file.stem.replace("_processed_converted", "")  # 从输入文件名

# out_path = Path(f"{input_file.parent}/{app_id}_prompt1_output.json")
# # out_path.parent.mkdir(parents=True, exist_ok=True)

# with out_path.open("w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)

# print("已保存:", out_path)

dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp
已保存: dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp\com.bandagames.mpuzzle.gp_prompt1_output.json


## 保存prompt2的结果

In [ ]:
# rs2 = '''
# {
# "app_id": "com.benoitletondor",
# "path_results": [
# {
# "path_change_id": "PC_001",
# "privacy_relevance": "none",
# "evidence_strength": "none",
# "event_types": [
# "NO_PRIVACY_RELEVANT_SIGNAL"
# ],
# "actions": [
# "none"
# ],
# "data_categories": [
# "none"
# ],
# "recipients": [
# "none"
# ],
# "purposes": [
# "none"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.95,
# "reasoning": "This path reflects removal of dynamic UI composition steps in MainActivity.onCreate, leaving only window access. The evidence points to startup/interface simplification rather than collection, processing, storage, sharing, consent, or rights-related behavior.",
# "uncertainties": [
# "The removed UI path could previously have supported privacy-relevant prompts or notices, but no such purpose is shown in the evidence.",
# "getWindow alone is not a meaningful privacy signal in this context."
# ]
# },
# {
# "path_change_id": "PC_002",
# "privacy_relevance": "medium",
# "evidence_strength": "strong_indirect",
# "event_types": [
# "EXTERNAL_ROUTING_SIGNAL",
# "NOTIFICATION_ACCESS_SIGNAL",
# "DEVICE_STATE_ACCESS_SIGNAL"
# ],
# "actions": [
# "route_to_external_component",
# "access_notification",
# "access_device_state"
# ],
# "data_categories": [
# "notification_content",
# "battery_status"
# ],
# "recipients": [
# "other_app",
# "operating_system",
# "unknown_external_recipient"
# ],
# "purposes": [
# "app_functionality",
# "system_integration",
# "unknown"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.7,
# "reasoning": "The old configuration/settings-driven startup flow was replaced with an asynchronous intent- and URI-based routing flow using chooser/package/class/data/extras APIs. References to notification-listener and battery-status receiver components provide indirect support that the new path may involve notification-related information and device-state integration. The evidence supports privacy-relevant routing and possible access signals, but it does not directly prove concrete personal-data collection, transfer, or disclosure content.",
# "uncertainties": [
# "Intent extras and URIs are present, but their payloads are unknown, so concrete data categories beyond indirect notification and battery/device-state signals cannot be confirmed.",
# "The notification-listener and battery-status references may be only component linkage rather than actual user-data use in this path.",
# "It is unclear whether routing stays fully internal, targets the operating system, or invokes third-party apps/services."
# ]
# },
# {
# "path_change_id": "PC_003",
# "privacy_relevance": "weak",
# "evidence_strength": "weak_indirect",
# "event_types": [
# "DATA_STORAGE_SIGNAL",
# "EXTERNAL_ROUTING_SIGNAL"
# ],
# "actions": [
# "store",
# "route_to_external_component"
# ],
# "data_categories": [
# "unknown_personal_data"
# ],
# "recipients": [
# "controller"
# ],
# "purposes": [
# "support",
# "app_functionality"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.56,
# "reasoning": "This path removes support-screen interactivity including startActivity/result passing and local storage-related APIs such as cache, no-backup files, and database helper configuration. That gives a weak indirect signal of reduced local data handling and reduced support-flow routing, but the evidence does not show what data, if any, was handled and does not directly establish privacy-specific operations.",
# "uncertainties": [
# "The storage-related APIs may have been used for non-personal UI/support artifacts rather than personal data.",
# "The removed activity/result paths may reflect generic navigation rather than privacy-relevant external routing.",
# "No direct evidence identifies any specific support request data, account data, or contact data."
# ]
# },
# {
# "path_change_id": "PC_004",
# "privacy_relevance": "none",
# "evidence_strength": "none",
# "event_types": [
# "NO_PRIVACY_RELEVANT_SIGNAL"
# ],
# "actions": [
# "none"
# ],
# "data_categories": [
# "none"
# ],
# "recipients": [
# "none"
# ],
# "purposes": [
# "none"
# ],
# "rights_related": [],
# "cross_border_related": false,
# "third_country_related": false,
# "sensitive_related": false,
# "biometric_related": false,
# "children_related": false,
# "confidence": 0.92,
# "reasoning": "The evidence shows splash/relaunch lifecycle simplification, removed UI traversal, and removal of timer cancellation on stop. No direct or indirect signal of personal-data collection, processing, storage, sharing, consent handling, or user-right handling is supported by these path changes.",
# "uncertainties": [
# "Lifecycle changes could theoretically affect background behavior duration, but no privacy-relevant data operation is evidenced here.",
# "The helper-branch substitution in splash initialization is too abstract to support a privacy interpretation."
# ]
# }
# ]
# }
# '''

In [ ]:
# import json
# from pathlib import Path

# # rs 是你现在那个三引号字符串
# data = json.loads(rs2)

# input_file = Path("dataset4geodiff\out_openai_sematic_geodiff_txt\com.benoitletondor.pixelminimalwatchface\com.benoitletondor.pixelminimalwatchface_processed_converted.json")         # 改成你的原始文件路径
# # print(input_file.parent)
# app_id = input_file.stem.replace("_processed_converted", "")  # 从输入文件名

# out_path = Path(f"{input_file.parent}/{app_id}_prompt2_output.json")
# # out_path.parent.mkdir(parents=True, exist_ok=True)

# with out_path.open("w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)

# print("已保存:", out_path)

In [ ]:
# rs = '''
# {
# "app_id": "com.bandagames.mpuzzle.gp",
# "path_clause_results": [
# {
# "path_change_id": "PC001",
# "bucket_predictions": [
# "NONE"
# ],
# "candidate_clauses": [],
# "final_clause_links": [],
# "rejected_clauses": [],
# "no_direct_match_reason": "This path only shows startup simplification and removal of generic initialization/system-service access, with no concrete notice, consent, rights, controller-contact, request-process, or specific data-handling clause signal."
# }
# ]
# }

# '''

In [ ]:
# import json
# from pathlib import Path

# # rs 是你现在那个三引号字符串
# data = json.loads(rs)

# input_file = Path("dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp\com.bandagames.mpuzzle.gp_processed_converted.json")         # 改成你的原始文件路径
# # print(input_file.parent)
# app_id = input_file.stem.replace("_processed_converted", "")  # 从输入文件名

# out_path = Path(f"{input_file.parent}/{app_id}_prompt2_output.json")
# # out_path.parent.mkdir(parents=True, exist_ok=True)

# with out_path.open("w", encoding="utf-8") as f:
#     json.dump(data, f, ensure_ascii=False, indent=2)

# print("已保存:", out_path)

已保存: dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp\com.bandagames.mpuzzle.gp_prompt2_output.json
